# 03 - Dunkelflaute and compound events

A Dunkelflaute is a *sustained* simultaneous shortfall of wind and solar
generation. Three choices govern how many you find, and all three must be
reported alongside any count:

- the thresholds,
- the minimum duration,
- the spatial aggregation scale.

The paper's reference definition: wind capacity factor below 10 %, solar
capacity factor below 5 %, sustained for at least 48 hours.

In [ ]:
import numpy as np
import pandas as pd
import xarray as xr

from alpinemet.indicators.dunkelflaute import (
    DUNKELFLAUTE_MIN_DURATION_HOURS,
    DUNKELFLAUTE_SOLAR_CF_THRESHOLD,
    DUNKELFLAUTE_WIND_CF_THRESHOLD,
    detect,
)

print(f"wind CF  < {DUNKELFLAUTE_WIND_CF_THRESHOLD:.0%}")
print(f"solar CF < {DUNKELFLAUTE_SOLAR_CF_THRESHOLD:.0%}")
print(f"for at least {DUNKELFLAUTE_MIN_DURATION_HOURS} h")

## A winter month with two calm, dull spells

In [ ]:
time = pd.date_range("2020-01-01", periods=24 * 30, freq="h")
rng = np.random.default_rng(11)

wind_cf = np.clip(rng.beta(2.0, 3.0, time.size), 0.0, 1.0)
solar_cf = np.clip(
    0.35 * np.sin(np.pi * (time.hour.to_numpy() - 8) / 9), 0.0, None
) * rng.uniform(0.4, 1.0, time.size)

# A 60 hour blocking episode and a 30 hour one.
wind_cf[200:260] = 0.03
solar_cf[200:260] *= 0.05
wind_cf[500:530] = 0.03
solar_cf[500:530] *= 0.05

series = {
    name: xr.DataArray(values, dims="time", coords={"time": time})
    for name, values in [("wind", wind_cf), ("solar", solar_cf)]
}

result = detect(series["wind"], series["solar"])
print(f"episodes detected : {int(result['event_count'])}")
print(f"hours in episodes : {float(result['total_hours']):.0f}")
print(f"longest episode   : {float(result['longest_event_hours']):.0f} h")

Only the 60 hour spell qualifies. The 30 hour one is a shortfall but not an
episode: its hours appear in `shortfall` and not in `dunkelflaute`.

In [ ]:
print(f"hours meeting the instantaneous condition: {int(result['shortfall'].sum())}")
print(f"hours inside a sustained episode        : {int(result['dunkelflaute'].sum())}")

## Threshold sensitivity

The paper notes that moving the wind threshold between 10 % and 20 % can change
event frequency by a factor of 3 to 5. Worth reproducing before quoting any
number.

In [ ]:
rows = []
for wind_threshold in [0.08, 0.10, 0.12, 0.15, 0.20]:
    for duration in [24, 48, 72]:
        run = detect(
            series["wind"],
            series["solar"],
            wind_threshold=wind_threshold,
            min_duration_hours=duration,
        )
        rows.append(
            {
                "wind_cf_threshold": wind_threshold,
                "min_duration_h": duration,
                "events": int(run["event_count"]),
                "hours": float(run["total_hours"]),
            }
        )

pd.DataFrame(rows).pivot(
    index="wind_cf_threshold", columns="min_duration_h", values="hours"
)

## Cold Dunkelflaute

Low generation coinciding with peak heating demand: the most stressful
combination for grid adequacy.

In [ ]:
temperature = xr.DataArray(
    5.0 - 10.0 * (np.arange(time.size) // 24 % 7 == 0), dims="time",
    coords={"time": time},
)
temperature[200:260] = -4.0

cold = detect(series["wind"], series["solar"], temperature=temperature)
print(f"Dunkelflaute hours      : {float(cold['total_hours']):.0f}")
print(f"Cold Dunkelflaute hours : {float(cold['cold_total_hours']):.0f}")

## Aggregation scale in complex terrain

During a persistent inversion, valley-floor installations can be in energy
drought while ridge-top turbines run near rated capacity. Whether that is a
system-level Dunkelflaute depends entirely on the scale of aggregation -- so
state the scale rather than letting the default decide.

In [ ]:
from alpinemet.indicators.compound import hellsturm

wind_field = xr.DataArray(
    np.zeros((48, 2, 2)) + 3.0,
    dims=("time", "latitude", "longitude"),
    coords={"time": time[:48], "latitude": [46.0, 47.0], "longitude": [11.0, 12.0]},
)
solar_field = xr.zeros_like(wind_field) + 100.0
# One exposed ridge cell: strong wind and full sun.
wind_field[:, 0, 0] = 18.0
solar_field[:, 0, 0] = 700.0

pointwise = hellsturm(wind_field, solar_field)
aggregated = hellsturm(wind_field, solar_field, aggregate=True)

print(f"grid-point-wise: {int(pointwise['hellsturm'].sum())} cell-hours detected")
print(f"domain-averaged: {int(aggregated['hellsturm'].sum())} hours detected")
print()
print("The ridge cell is in a Hellsturm throughout; the domain mean never is.")